[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/16_PID_Control.ipynb)

# DiveLab

## Notebook 16 — PID Control: P, I and D as Diver Behavior

**Guiding question:** How do proportional, integral and derivative actions change the way a diver corrects depth error?

In DiveLab, the controller is the diver.

PID control gives us a compact mathematical language for three familiar control ideas:

- react to the **current error**;
- remember the **accumulated error**;
- react to the **trend**.

We will build the controller progressively:

\[
P \rightarrow PI \rightarrow PD \rightarrow PID
\]

## Learning objectives

By the end of this notebook, you will be able to:

- explain P, I and D in simple words;
- interpret PID as a model of diver correction behavior;
- derive the PID transfer function;
- see how proportional action changes responsiveness;
- understand how integral action removes persistent error;
- understand how derivative action adds damping;
- identify overshoot and oscillation;
- understand actuator saturation;
- explain integral windup and anti-windup;
- connect PID tuning with poles and closed-loop dynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. The control error

Suppose the target depth is:

\[
z_r.
\]

Define error as:

\[
e(t)=z(t)-z_r(t).
\]

With our convention:

- positive \(e\): diver is too deep;
- negative \(e\): diver is too shallow.

The controller chooses a buoyancy-control action \(u(t)\) from this error.

# 2. Proportional control

The simplest controller is:

\[
\boxed{u(t)=K_Pe(t)}
\]

In simple words:

> the larger the current error, the larger the correction.

For the diver:

- too deep → add buoyancy;
- too shallow → reduce buoyancy.

The correction depends only on what is wrong **now**.

## Diver interpretation of \(P\)

A proportional diver-controller behaves like:

> "I am 1 m too deep, so I make a moderate correction.  
> I am 3 m too deep, so I make a larger correction."

The gain \(K_P\) determines how aggressively error is corrected.

# 3. A simple controlled plant

To isolate PID ideas, first use a simple second-order vertical plant:

\[
\ddot z + c\dot z - \alpha^2 z = -b u.
\]

The term:

\[
-\alpha^2 z
\]

represents the local buoyancy instability.

We work in depth deviation from the target, so the desired state is:

\[
z=0.
\]

In [ ]:
alpha = 0.20
c = 0.20
b = 1.0

The uncontrolled characteristic polynomial is:

\[
s^2 + cs - \alpha^2.
\]

Because the constant term is negative, one pole lies in the right half-plane.

The plant is unstable.

In [ ]:
open_loop_poles = np.roots([1, c, -alpha**2])
print("Open-loop poles:", open_loop_poles)

# 4. Simulate proportional control

Use:

\[
u=K_P z.
\]

Then:

\[
\ddot z + c\dot z - \alpha^2z = -bK_Pz.
\]

So:

\[
\ddot z + c\dot z + (bK_P-\alpha^2)z=0.
\]

Proportional feedback can create a restoring term if:

\[
bK_P>\alpha^2.
\]

In [ ]:
def simulate_controller(
    Kp=0.0,
    Ki=0.0,
    Kd=0.0,
    z0=1.0,
    v0=0.0,
    duration=30.0,
    dt=0.005,
    disturbance=0.0,
    u_limit=None,
    anti_windup=False,
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    integ = np.zeros(n)
    u = np.zeros(n)

    z[0] = z0
    v[0] = v0

    for k in range(n - 1):
        e = z[k]

        # Since depth velocity is dz/dt = -v, derivative of error is -v
        de_dt = -v[k]

        integ_candidate = integ[k] + e * dt

        u_unsat = (
            Kp * e
            + Ki * integ_candidate
            + Kd * de_dt
        )

        if u_limit is None:
            uk = u_unsat
            integ[k + 1] = integ_candidate
        else:
            uk = np.clip(u_unsat, -u_limit, u_limit)

            if anti_windup and uk != u_unsat:
                integ[k + 1] = integ[k]
            else:
                integ[k + 1] = integ_candidate

        # Equation:
        # z'' + c z' - alpha^2 z = -b u + disturbance
        # z' = -v, therefore z'' = -v'
        # => v' = c z' - alpha^2 z + b u - disturbance
        # with z' = -v:
        dv = -c * v[k] - alpha**2 * z[k] + b * uk - disturbance

        v[k + 1] = v[k] + dv * dt
        z[k + 1] = z[k] - v[k + 1] * dt

        u[k] = uk

    u[-1] = u[-2]
    integ[-1] = integ[-2]

    return t, z, v, integ, u

In [ ]:
Kp_values = [0.03, 0.08, 0.20]

for Kp in Kp_values:
    t, z, v, integ, u = simulate_controller(Kp=Kp)
    plt.plot(t, z, label=f"Kp={Kp}")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth error")
plt.title("Proportional control")
plt.grid(True)
plt.legend()
plt.show()

# 5. What proportional gain changes

Increasing \(K_P\) generally increases restoring strength.

But more proportional gain does not automatically mean better behavior.

Depending on the plant, high gain can produce:

- faster response;
- overshoot;
- oscillation;
- sensitivity to delay and noise.

This reconnects with earlier DiveLab notebooks.

# 6. Persistent disturbance

Suppose there is a constant external disturbance.

For example, our simplified model may contain an unmodeled constant force.

A proportional controller may settle with a nonzero error because it needs some persistent error to generate the control action that balances the disturbance.

In [ ]:
t_p, z_p, v_p, integ_p, u_p = simulate_controller(
    Kp=0.12,
    disturbance=0.03
)

plt.plot(t_p, z_p)
plt.axhline(0, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth error")
plt.title("P control with a constant disturbance")
plt.grid(True)
plt.show()

print("Final error:", z_p[-1])

# 7. Integral control

Integral action uses the accumulated error:

\[
\boxed{
u_I(t)=K_I\int_0^t e(\tau)\,d\tau
}
\]

In simple words:

> if the same error keeps persisting, the controller gradually increases its correction.

Integral action remembers the past.

## Diver interpretation of \(I\)

Imagine the diver remains slightly too deep for several seconds.

A purely proportional strategy may continue applying the same modest correction.

Integral action behaves more like:

> "I have been too deep for too long. My previous correction was not enough, so I increase it."

This accumulated memory is useful for eliminating persistent offsets.

# 8. PI control

In [ ]:
t_pi, z_pi, v_pi, integ_pi, u_pi = simulate_controller(
    Kp=0.12,
    Ki=0.015,
    disturbance=0.03
)

plt.plot(t_p, z_p, label="P")
plt.plot(t_pi, z_pi, label="PI")
plt.axhline(0, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth error")
plt.title("Integral action removes persistent error")
plt.grid(True)
plt.legend()
plt.show()

print("P final error :", z_p[-1])
print("PI final error:", z_pi[-1])

Integral action can remove steady-state error.

But it also adds dynamics.

Too much integral action can produce:

- overshoot;
- slow oscillation;
- large accumulated control effort.

So \(I\) is powerful but must be tuned carefully.

# 9. Derivative control

Derivative action is:

\[
\boxed{
u_D(t)=K_D\frac{de}{dt}
}
\]

It reacts not only to the error, but to how quickly the error is changing.

For constant target depth:

\[
e=z-z_r,
\]

so:

\[
\dot e=\dot z=-v.
\]

## Diver interpretation of \(D\)

Derivative action is naturally interpreted as reacting to **vertical trend**.

The diver may be close to the desired depth, but rising quickly.

A purely proportional controller sees only a small depth error.

A derivative-aware controller recognizes:

> "The error is small now, but I am moving in the wrong direction quickly."

This anticipatory effect adds damping.

# 10. PD control

In [ ]:
t_p2, z_p2, v_p2, integ_p2, u_p2 = simulate_controller(
    Kp=0.20,
    Kd=0.0
)

t_pd, z_pd, v_pd, integ_pd, u_pd = simulate_controller(
    Kp=0.20,
    Kd=0.8
)

plt.plot(t_p2, z_p2, label="P")
plt.plot(t_pd, z_pd, label="PD")
plt.axhline(0, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth error")
plt.title("Derivative action adds damping")
plt.grid(True)
plt.legend()
plt.show()

Derivative action often reduces overshoot and oscillation because it reacts to motion before the position error becomes large.

# 11. PID control

Combine all three terms:

\[
\boxed{
u(t)
=
K_Pe(t)
+
K_I\int_0^t e(\tau)d\tau
+
K_D\frac{de}{dt}
}
\]

In Laplace form:

\[
\boxed{
C(s)
=
K_P
+
\frac{K_I}{s}
+
K_Ds
}
\]

The three terms have different roles:

### P — present

\[
K_Pe
\]

reacts to current error.

### I — past

\[
K_I\int e\,dt
\]

reacts to accumulated error.

### D — trend

\[
K_D\dot e
\]

reacts to how the error is changing.

A memorable summary is:

\[
\boxed{
\text{P = present,\quad I = past,\quad D = trend}
}
\]

# 12. PID simulation

In [ ]:
t_pid, z_pid, v_pid, integ_pid, u_pid = simulate_controller(
    Kp=0.18,
    Ki=0.010,
    Kd=0.9,
    disturbance=0.03
)

plt.plot(t_pid, z_pid)
plt.axhline(0, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth error")
plt.title("PID control with constant disturbance")
plt.grid(True)
plt.show()

A well-tuned PID controller can combine:

- fast correction from P;
- zero steady-state error from I;
- damping from D.

But tuning involves tradeoffs.

# 13. Compare P, PI, PD and PID

In [ ]:
configs = {
    "P":   dict(Kp=0.18, Ki=0.0,   Kd=0.0),
    "PI":  dict(Kp=0.18, Ki=0.010, Kd=0.0),
    "PD":  dict(Kp=0.18, Ki=0.0,   Kd=0.9),
    "PID": dict(Kp=0.18, Ki=0.010, Kd=0.9),
}

for name, cfg in configs.items():
    t_c, z_c, v_c, i_c, u_c = simulate_controller(
        **cfg,
        disturbance=0.03
    )
    plt.plot(t_c, z_c, label=name)

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth error")
plt.title("P vs PI vs PD vs PID")
plt.grid(True)
plt.legend()
plt.show()

# 14. PID and pole movement

For the simplified plant:

\[
\ddot z+c\dot z-\alpha^2z=-bu,
\]

with PD control:

\[
u=K_Pz+K_D\dot z,
\]

the closed-loop characteristic polynomial becomes:

\[
s^2+(c+bK_D)s+(bK_P-\alpha^2).
\]

This makes the roles clear:

- \(K_P\) changes restoring strength;
- \(K_D\) changes damping.

In [ ]:
def pd_poles(Kp, Kd):
    return np.roots([
        1,
        c + b*Kd,
        b*Kp - alpha**2
    ])

for Kp, Kd in [(0.08,0.0),(0.18,0.0),(0.18,0.9)]:
    print(f"Kp={Kp}, Kd={Kd} -> poles={pd_poles(Kp,Kd)}")

# 15. Integral action increases system order

Integral control introduces a new state:

\[
x_I(t)=\int_0^t e(\tau)d\tau.
\]

Therefore:

\[
\dot x_I=e.
\]

The controller now has memory, and the closed-loop system has an additional dynamical state.

This is why adding integral action changes the characteristic polynomial order.

# 16. Actuator saturation

A real diver cannot add or vent gas infinitely quickly.

So:

\[
|u|\le u_{\max}.
\]

If PID demands more than the actuator can provide, the command saturates.

In [ ]:
u_limit = 0.12

t_sat, z_sat, v_sat, integ_sat, u_sat = simulate_controller(
    Kp=0.30,
    Ki=0.05,
    Kd=0.7,
    z0=2.0,
    u_limit=u_limit
)

plt.plot(t_sat, u_sat)
plt.axhline(u_limit, linestyle="--")
plt.axhline(-u_limit, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Control input")
plt.title("Actuator saturation")
plt.grid(True)
plt.show()

# 17. Integral windup

During saturation, the integral term may continue accumulating error:

\[
\int e\,dt.
\]

But the actuator cannot deliver the requested action.

The stored integral can become very large.

When the system finally leaves saturation, this accumulated integral may drive excessive correction.

This phenomenon is called **integral windup**.

In [ ]:
plt.plot(t_sat, integ_sat)

plt.xlabel("Time [s]")
plt.ylabel("Integral state")
plt.title("Integral accumulation during saturation")
plt.grid(True)
plt.show()

# 18. Anti-windup

A simple anti-windup strategy is:

> if the actuator is saturated, temporarily stop accumulating the integral term.

This is not the only anti-windup method, but it is easy to understand.

In [ ]:
t_aw, z_aw, v_aw, integ_aw, u_aw = simulate_controller(
    Kp=0.30,
    Ki=0.05,
    Kd=0.7,
    z0=2.0,
    u_limit=u_limit,
    anti_windup=True
)

plt.plot(t_sat, z_sat, label="Without anti-windup")
plt.plot(t_aw, z_aw, label="With anti-windup")
plt.axhline(0, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth error")
plt.title("Effect of anti-windup")
plt.grid(True)
plt.legend()
plt.show()

# 19. Derivative action and sensor noise

Derivative action responds strongly to rapid changes.

Unfortunately, measurement noise often contains high-frequency fluctuations.

Therefore derivative control can amplify sensor noise.

This reconnects with Notebook 06.

In practice, derivative action is often filtered rather than implemented as an ideal:

\[
K_Ds.
\]

A common filtered derivative is:

\[
C_D(s)
=
K_D
\frac{Ns}{s+N}.
\]

At low frequency it behaves approximately like a derivative.

At very high frequency, its gain is limited.

# 20. Derivative on measurement

Another practical detail:

If the reference changes suddenly, differentiating the error can create a large **derivative kick**.

For this reason, practical PID implementations often apply derivative action to the measured output rather than directly to the error.

This distinction becomes important in real control systems.

# 21. PID block structure

A conceptual PID controller can be drawn as:

```text
                +---- P ----+
error ----------+---- I ----+----> control action
                +---- D ----+
```

Each branch sees the same error but responds to a different temporal aspect:

- present magnitude;
- accumulated history;
- rate of change.

# 22. Closed-loop transfer function with PID

For plant:

\[
G(s)
\]

and PID controller:

\[
C(s)
=
K_P+\frac{K_I}{s}+K_Ds,
\]

the closed-loop transfer function is:

\[
\boxed{
T(s)
=
\frac{C(s)G(s)}
{1+C(s)G(s)}
}
\]

The PID gains reshape the closed-loop poles and zeros.

# 23. A simple first-order plant with PI control

Take:

\[
G(s)=\frac{1}{\tau s+1}
\]

and:

\[
C(s)=K_P+\frac{K_I}{s}.
\]

The controller introduces a pole at:

\[
s=0
\]

through the integral term.

It also introduces a zero at:

\[
s=-\frac{K_I}{K_P}.
\]

So PI control changes both poles and zeros of the closed-loop system.

# 24. Tuning is a tradeoff

Typical tuning objectives include:

- short rise time;
- small overshoot;
- short settling time;
- small steady-state error;
- limited control effort;
- robustness to noise and delay.

Improving one objective can degrade another.

PID tuning is therefore an engineering compromise.

# 25. Diver interpretation of tuning

In human terms:

### Too little gain

Corrections are weak and slow.

### Too much proportional gain

Corrections can become aggressive.

### Too much integral action

The diver may keep "adding correction" because past error remains accumulated.

### Too much derivative action

The controller becomes overly sensitive to rapid fluctuations and noisy trend estimates.

The analogy is useful because it connects control theory with observable behavior.

# 26. A synthetic tuning experiment

Let's compare several PID settings.

In [ ]:
tunings = {
    "gentle":     dict(Kp=0.10, Ki=0.004, Kd=0.5),
    "balanced":   dict(Kp=0.18, Ki=0.010, Kd=0.9),
    "aggressive": dict(Kp=0.35, Ki=0.030, Kd=1.2),
}

for name, cfg in tunings.items():
    t_c, z_c, v_c, i_c, u_c = simulate_controller(
        **cfg,
        disturbance=0.03
    )
    plt.plot(t_c, z_c, label=name)

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth error")
plt.title("PID tuning changes closed-loop behavior")
plt.grid(True)
plt.legend()
plt.show()

# 27. Performance metrics

We can quantify the response with:

- maximum absolute error;
- settling behavior;
- integral absolute error;
- total control effort.

In [ ]:
def metrics(t, z, u):
    dt = t[1] - t[0]

    return {
        "max_abs_error": np.max(np.abs(z)),
        "IAE": np.sum(np.abs(z)) * dt,
        "control_effort": np.sum(np.abs(u)) * dt,
    }

for name, cfg in tunings.items():
    t_c, z_c, v_c, i_c, u_c = simulate_controller(
        **cfg,
        disturbance=0.03
    )

    print(name)
    for key, value in metrics(t_c, z_c, u_c).items():
        print(f"  {key:16s}: {value:.4f}")

# 28. Where frequency response enters

So far we tuned PID mostly from time-domain behavior.

But PID also changes how the system reacts to disturbances at different frequencies.

For example:

- slow drift;
- breathing oscillations;
- rapid sensor noise.

To understand this systematically, we need **frequency response**.

That is the subject of Notebook 17.

# Exercises

### 1. Proportional gain

Set:

```python
Ki = 0
Kd = 0
```

and vary \(K_P\).

Find the smallest \(K_P\) that stabilizes the simplified plant.

### 2. Integral action

Add a constant disturbance.

Compare P and PI control.

How does \(K_I\) affect steady-state error and overshoot?

### 3. Derivative action

Use PD control.

Increase \(K_D\) gradually.

How do the poles move?

In [ ]:
# Your code here

### 4. Windup

Use a large initial error and a small actuator limit.

Compare PID with and without anti-windup.

### 5. Noise sensitivity

Add noise to the depth measurement.

Estimate the derivative numerically.

What happens to the derivative control signal?

# Challenge — tune a PID controller

Choose \(K_P,K_I,K_D\) to obtain a response with:

- low overshoot;
- small steady-state error;
- moderate control effort.

Define your own quantitative cost function, for example:

\[
J
=
w_1\,IAE
+
w_2\,u_{\text{effort}}
+
w_3\,\text{overshoot}.
\]

Search over a grid of PID gains and find a good compromise.

In [ ]:
# Your code here

# Summary

In this notebook we built PID control progressively.

We learned:

### Proportional

\[
u_P=K_Pe
\]

reacts to the current error.

### Integral

\[
u_I
=
K_I\int e\,dt
\]

reacts to accumulated error and can remove persistent offset.

### Derivative

\[
u_D=K_D\dot e
\]

reacts to trend and can add damping.

Together:

\[
\boxed{
u
=
K_Pe
+
K_I\int e\,dt
+
K_D\dot e
}
\]

We also studied:

- pole movement;
- damping;
- actuator saturation;
- integral windup;
- anti-windup;
- derivative noise sensitivity;
- tuning tradeoffs.

### Diver interpretation

\[
\boxed{
P=\text{current error}
}
\]

\[
\boxed{
I=\text{memory of persistent error}
}
\]

\[
\boxed{
D=\text{trend / vertical motion}
}
\]

### Next — Notebook 17

We now move to **Frequency Response**.

We will ask:

> How does the diver-control system react to slow and fast disturbances?

This will introduce:

- sinusoidal inputs;
- gain and phase;
- Bode plots;
- bandwidth;
- phase margin;
- breathing as a periodic disturbance;
- delay as phase lag.